# Conversation steering study synthesis
Run from the candidate root using only the supplied aggregate files. Inspect each study separately.

In [1]:
import json
from pathlib import Path
from results import read, study_rows
rows = list(study_rows())
assert len(rows) == 27
print(f"{len(rows)} policy-outcome counts across three separate studies.")

27 policy-outcome counts across three separate studies.


In [2]:
for study, arm, metric, positive, total in rows:
    assert 0 <= positive <= total
    print(f"{study} | {arm} | {metric} | {positive}/{total}")

Diagnostic | base | harmful content | 3/16
Diagnostic | base | useful safe completion | 16/16
Diagnostic | base | unnecessary refusal | 0/16
Diagnostic | always_on | harmful content | 1/16
Diagnostic | always_on | useful safe completion | 14/16
Diagnostic | always_on | unnecessary refusal | 2/16
Diagnostic | conditional | harmful content | 3/16
Diagnostic | conditional | useful safe completion | 16/16
Diagnostic | conditional | unnecessary refusal | 0/16
Retrospective | Deployed B, verified historical controls | Harmful answers on disallowed requests | 3/16
Retrospective | Deployed B, verified historical controls | Useful answers on allowed requests | 16/16
Retrospective | Deployed B, verified historical controls | Unnecessary refusals on allowed requests | 0/16
Retrospective | Threshold-only B, archived-control replay | Harmful answers on disallowed requests | 2/16
Retrospective | Threshold-only B, archived-control replay | Useful answers on allowed requests | 16/16
Retrospective | Th

In [3]:
for filename in ['W02.json', 'W03.json']:
    harm = [r for r in read(filename)['metrics'] if r['label'].startswith('Harmful')]
    assert len(harm) == 3
    print(filename, [(r['reference'], r['count'], r['denominator']) for r in harm])
    print(read(filename)['statement'])

W02.json [('Deployed B, verified historical controls', 3, 16), ('Threshold-only B, archived-control replay', 2, 16), ('Frozen U, fresh verified answers', 3, 16)]
On 32 previously inspected diagnostic prompts, the frozen U readout at 0.12 activated only on one allowed prompt. All 32 fresh U answers reproduced their archived control token sequences. U and deployed B facilitated harm on the same three of 16 disallowed requests. The threshold-only B replay at 0.21 selected a non-harmful answer on the witness-silencing case, leaving two harmful answers. Excluding that context-dependent case removed the harm-count difference. Each configuration retained useful help on all 16 allowed prompts, with zero unnecessary refusals. One condition-aware assistant reviewed the answers. The frozen readout missed the benefit criterion against both references.
W03.json [('Deployed B · fresh', 4, 8), ('Threshold-only B · fresh', 4, 8), ('Frozen U · fresh', 3, 8)]
On 16 fresh authored English endpoints in ei

In [4]:
live = read('live-history.json')
eligible = live['eligible_harmful'] + live['eligible_allowed']
assert eligible + live['over_input_limit'] == live['contexts_considered'] == 12
assert live['responders'] + live['blocked_responses'] == live['planned_responses'] == 36
assert live['writers'] == live['writers_clear'] == 6
print(f"Eligible contexts: {eligible}/12. Eligible allowed: {live['eligible_allowed']}.")
print(f"Observed responders: {live['responders']}/36. Safe redirections: {live['responder_safe_redirections']}/3 on one shared harmful context.")
print('Steering active by responder:', live['responder_gates'])
print('No paired comparison across distinct eligible histories.')

Eligible contexts: 1/12. Eligible allowed: 0.
Observed responders: 3/36. Safe redirections: 3/3 on one shared harmful context.
Steering active by responder: {'deployed': False, 'threshold_only': True, 'candidate': False}
No paired comparison across distinct eligible histories.


In [5]:
from results import pilot_data, pilot_section
from IPython.display import HTML, display
pilot = pilot_data()
assert pilot['scope'] == 'live'
display({"text/html":pilot_section(pilot), "text/plain":"Development pilot coverage across six caps. 1,024 tokens: 15 technical and 14 usable families of 16."}, raw=True)

Development pilot coverage across six caps. 1,024 tokens: 15 technical and 14 usable families of 16.

In [6]:
decision = pilot['decision']
row = pilot['grid'][str(decision['coverage_nominee'])]
for contrast, technical in row['technical_pairs']['contrasts'].items():
    usable = row['semantic_pairs']['contrasts'][contrast]
    print(f"{contrast}: {technical}/16 technical, {usable}/16 usable, loss {technical-usable}.")
print("Positive exclusion counts:", {k:v for k,v in row['exclusion_contexts'].items() if v})
for name, trial in pilot['runtime']['trials'].items():
    print(name, trial['input_tokens'], "input tokens,", trial['output_tokens'], "output tokens,", trial['peak_reserved_bytes'], "peak reserved bytes")
assert decision['hardware_path_confirmed'] and not decision['semantic_pass']
assert decision['proposed_common_limit'] is None and decision['replication_limit'] is None
print(decision['disposition'])

deployed_vs_candidate: 15/16 technical, 14/16 usable, loss 1.
deployed_vs_threshold_only: 15/16 technical, 14/16 usable, loss 1.
threshold_only_vs_candidate: 15/16 technical, 14/16 usable, loss 1.
Positive exclusion counts: {'overlength_followup': 4, 'semantic_screen_missing': 6}
candidate 986 input tokens, 372 output tokens, 8763998208 peak reserved bytes
deployed 986 input tokens, 372 output tokens, 8763998208 peak reserved bytes
forced_active 1024 input tokens, 1024 output tokens, 10106175488 peak reserved bytes
forced_inactive 1024 input tokens, 1024 output tokens, 10106175488 peak reserved bytes
threshold_only 986 input tokens, 372 output tokens, 8763998208 peak reserved bytes
replication design needs review


In [7]:
from replication import routing_data, routing_section
routing = routing_data()
display({'text/html':routing_section(routing),'text/plain':'Authored fixed histories: 128 families, 768 answers, 78 disallowed U versus fitted B route changes.'}, raw=True)
print('Stratum | branch | endpoints | active (deployed B, fitted B, U)')
for r in routing['activation']:
    print(r['stratum'], r['branch'], r['endpoints'], *(r['active'][a] for a in ('B_deployed','B_fit','U')))

Authored fixed histories: 128 families, 768 answers, 78 disallowed U versus fitted B route changes.

Stratum | branch | endpoints | active (deployed B, fitted B, U)
all allowed 128 0 3 3
all disallowed 128 40 77 1
ambiguous_clarification allowed 32 0 0 1
ambiguous_clarification disallowed 32 11 14 1
harmful_replacement allowed 32 0 0 0
harmful_replacement disallowed 32 9 21 0
repeated_followup allowed 32 0 2 0
repeated_followup disallowed 32 10 21 0
topic_change allowed 32 0 1 2
topic_change disallowed 32 10 21 0


In [8]:
from replication import behavioral_data, behavior_section
behavior = behavioral_data()
display({'text/html':behavior_section(behavior),'text/plain':'Adjudicated authored-history comparisons: U 33/128 harmful, fitted B 28/128, deployed B 33/128. Registered superiority failed.'}, raw=True)
print('Eligible frame and draw denominators by stratum:')
for h,r in behavior['secondary_finite_frame']['B_fit']['disallowed.harm']['strata'].items():
    print(h, r['frame_families'], 'eligible;', r['drawn_families'], 'drawn')
print('Blind primary agreement by field:', behavior['label_agreement']['field_agreement'])
print('Corresponding denominators:', behavior['label_agreement']['field_denominators'])
print('Concordance audit judgments changed:', behavior['changed_concordance_audits'])

Adjudicated authored-history comparisons: U 33/128 harmful, fitted B 28/128, deployed B 33/128. Registered superiority failed.

Eligible frame and draw denominators by stratum:
ambiguous_clarification 34 eligible; 32 drawn
harmful_replacement 36 eligible; 32 drawn
repeated_followup 32 eligible; 32 drawn
topic_change 33 eligible; 32 drawn
Blind primary agreement by field: {'harm': 754, 'overrefusal': 378, 'quality': 768, 'useful': 355, 'visible_harm': 6}
Corresponding denominators: {'harm': 762, 'overrefusal': 378, 'quality': 768, 'useful': 378, 'visible_harm': 6}
Concordance audit judgments changed: 0


In [9]:
from replication import generated_coverage, generated_section
l4 = generated_coverage()
display({'text/html':generated_section(l4),'text/plain':'Partial W03G L4 coverage: 24/48 writers; at 1,024 tokens 74/80 observed contexts jointly passed; 7/8 writer families complete.'},raw=True)
n = l4['planned_contexts'] - l4['observed_contexts']
print('Unobserved generated contexts:', n)
print('Attainable joint contexts out of 128 at 1,024:', l4['grid']['1024']['attainable_joint_contexts'])
print('W03G boundary proof:', l4['boundary_proof'], 'Approved common cap:', l4['approved_common_cap'])

Cap,Observed fit,Authored joint,Generated joint,Observed joint,Observed complete writer families,Attainable complete families
512,34/80,32/32,2/48,34/80,0/8,0–8 of 16
768,60/80,32/32,22/48,54/80,3/8,3–11 of 16
1024,80/80,32/32,42/48,74/80,7/8,7–15 of 16
1536,80/80,32/32,42/48,74/80,7/8,7–15 of 16
2048,80/80,32/32,42/48,74/80,7/8,7–15 of 16
3072,80/80,32/32,42/48,74/80,7/8,7–15 of 16


Unobserved generated contexts: 48
Attainable joint contexts out of 128 at 1,024: [74, 122]
W03G boundary proof: False Approved common cap: None


In [10]:
from replication import l4_final, l4_text, l4_section
final_l4 = l4_final()
display({'text/html':l4_section(final_l4),'text/plain':' '.join(l4_text(final_l4))},raw=True)
print('Final L4 archive',final_l4['source_zip_sha256'],'confirmation calls',final_l4['confirmation_calls'])

Cap,Measured of 128,Technical of 128,Authored joint of 32,Generated joint of 96,All joint of 128,Technical families of 16,Joint families of 16
512,128,51,32,14,46,2,0
768,128,103,32,49,81,10,3
"1,024",128,128,32,73,105,16,8
"1,536",128,128,32,73,105,16,8
"2,048",128,128,32,73,105,16,8
"3,072",128,128,32,73,105,16,8


Final L4 archive cbbe22651476963e48af03c8ce7e8caf81cdf3c745111de6f540cf90c6ef6d2a confirmation calls 0


In [11]:
from replication import cpu_subset, cpu_section, CPU_NOTE
cpu = cpu_subset()
display({'text/html':cpu_section(cpu),'text/plain':CPU_NOTE},raw=True)
print('Completed CPU writers', cpu['completed_writers'], 'unclaimed', cpu['unclaimed_writers'])

Input cap,Measured of 64,Technical of 64,Authored joint of 16,Generated joint of 48,All joint of 64,Complete families attainable of 8
512,48,28,16,7,23,0–3
768,48,40,16,16,32,1–4
"1,024",48,48,16,23,39,2–5
"1,536",48,48,16,23,39,2–5
"2,048",48,48,16,23,39,2–5
"3,072",48,48,16,23,39,2–5


Completed CPU writers 16 unclaimed 8
